# Finding the Earth-Sun Lagrange Points

This set of exercises explores the so-called restricted three-body problem where a small object, such as an asteroid or satellite, is interacting with two large objects, such as the Sun and the Earth, through Newtonian gravity. Specifically, this exercise set looks at five special locations where a small object can orbit synchronously with the Earth. These locations are called Lagrange points, and they can be found using Newton's law of gravitation and Newton's second law of motion. For any two-body system there exist five Lagrange points. Three of them can be found by finding the zeros of a single non-linear equation using Newton's method (also known as the Newton-Raphson method). The last two Lagrange points can be found by finding the zeros of a pair of coupled non-linear equations using the vector version of Newton's method.

Through completing this exercise, students will be able to:
* Use Newtonian mechanics to derive the equations of motion for a small object moving in a 2D plane under the influence of two gravitating bodies (Exercise 1)
* Visualize the "landscape" of a multi-dimensional root-finding problem (Exercise 2)
* Use Newton's method for finding zeros of both a single equation and a system of coupled equations. (Exercises 3 and 4)
* Use an optimized root-finding algorithm from `SciPy.Optimize`, to solve a non-linear system of equations faster and with greater accuracy. (Exercise 5)
* Explore the concept of stable and unstable equilibrium in physical systems and use the Jacobian matrix to determine the stability of an equilibrium (or psuedo-equilibrium) point. (Exercise 6)

## Theory
<div align="center">
    <img src="https://raw.githubusercontent.com/jstupak/ComputationalPhysics/master/Images/JWST_med.jpg" style="height:250px"/><br>
        <figcaption>JWST</figcaption>
</div>

Suppose you're NASA and you want to find a nice place to park a telescope such as the James Webb Space Telescope (JWST), pictured above. You could put in orbit around the Earth, but then you have a big, bright planet covering a big patch of your sky, not to mention all the other bits of space junk zooming around. You could put it in orbit around the moon, except then you have the same problem but with a moon. You could send your telescope completely out of Earth's orbit so that it orbits the Sun, but then it would steadily move farther and farther away over time.

The solution to NASA's problem of where to park a big telescope like JWST is a special place called a Lagrange point. Lagrange points are locations where a small body can rotate synchronously with two large bodies in any stable gravitational orbit. That means that NASA can send JWST to one of the Earth-Sun Lagrange points and it will orbit the Sun synchronously with the Earth, so that from our vantage point JWST won't move at all. This gets JWST away from Earth and all its light and space junk, but allows it to move along with Earth for as long as the telescope lasts. There are five such locations for any two-body system, and they are labeled L1 through L5, as shown in the diagram below. Lagrange points exist for elliptical orbits as well as circular ones, but for these exercises we'll stick with circular orbits.

<div align="center">
    <img src="https://raw.githubusercontent.com/jstupak/ComputationalPhysics/master/Images/Coordinate_System.png" style="height:250px"/><br>
        <figcaption>Lagrange Points</figcaption>
</div>

The first three Lagrange points were discovered by Swiss mathematician Leonhard Euler. The last two were found by Italian mathematician Joseph Louis Lagrange. Lagrange thoroughly explored the so-called restricted three body problem, where one of the three objects is assumed to have a very small mass compared to the other two.

Solving for the location of the Lagrange points requires solving either a single non-linear equation or a system of two non-linear equations. This can be done using Newton's method or its vector generalization. 

To solve a single non-linear equation using Newton's method, we begin by putting the equation in the form $f(x)=0.$  We then need to find some value of $x$ for which $f(x)$ evaluates to zero.  This is referred to as "root finding."  To find the roots, we begin with a reasonable guess and iterate until our sequence of guesses (hoprefully) converges to a root.  At each iteration, the subsequent guess $x_{n+1}$ is calculated from the current guess $x_n$ via

$$ x_{n+1} = x_n - \frac{ f(x_n) }{ f^\prime (x_n) }, $$

where $ f^\prime (x_n) = \left. \frac{ d f }{ d x } \right|_{x_n} $. Through repeated application of this procedure, the value of $f(x)$ should converge to zero, at which point we can identify our final guess as a root of $f(x)$.  In other words, we have solved the non-linear equation.

Newton's method for solving a single equation can be generalized to find the solutions to a system of coupled equations. Suppose we have two equations of the form 
$$ g(x, y)=0 $$
$$ h(x, y)=0 $$
These can be re-written in vector form as 
$$ \vec{F}(\vec{z}) = \left[ g (x,y), h(x,y)\right] = \left[ 0, 0 \right]$$
where $\vec{z} = \left[x, y \right]$. 
As in the scalar case, to use Newton's method we need a derivative, but now there are four possible first derivatives $\partial g / \partial x$, $\partial g / \partial y$, $\partial h / \partial x$, and $\partial h / \partial y$. So which derivative do we need? The answer is all of them. 

A Jacobian matrix is defined as a matrix that contains the derivatives of all our functions with respect to all their variables. In our case the Jacobian matrix would look like
$$ \mathbf{J} = \left[\begin{array}
{rr}
\frac{ \partial g }{ \partial x} & \frac{ \partial g }{ \partial y}  \\
\frac{ \partial h }{ \partial x} & \frac{ \partial h }{ \partial y}
\end{array}\right]\tag{1}
$$
One additional complication is that in Newton's method for a scalar we can simply divide $f(x)$ by the value of its derivative at a point, but we can't divide the vector $\vec{F}(\vec{z})$ by the Jacobian. Instead, we can multiply the vector $\vec{F}(\vec{z})$ by the inverse of the Jacobian matrix evaluated at a point. Therefore, Newton's method for solving a system of coupled equations updates the guess $\vec{z}_n$ at each iteration according to:
$$ \vec{z}_{n+1} = \vec{z}_n - \mathbf{J}^{-1} (\vec{z}_n) \vec{F} (\vec{z}_n).\tag{2}$$

Note that the $\mathbf{J}^{-1} (\vec{z}_n) \vec{F} (\vec{z}_n)$ is the multiplication of a matrix by a vector, which yields back a vector. In the same way as the scalar case, this version of Newton's method requires an initial guess, and then each time we iterate through the formula our guess will be updated and *hopefully* converge to the values of $x$ and $y$ that satisfy our equations.

One other point of note: Lagrange points are not true equilibrium points. Objects at those locations experience an acceleration, however the acceleration is such that they can orbit the Sun with exactly the same period as the Earth, thus maintaining a fixed location relative to the Sun and Earth. Another way to think about this is to consider a rotating reference frame, in which the Lagrange points are true equilibrium points. In either case, Lagrange points can be identified as stable or unstable using the eigenvalues of the Jacobian matrix as if they were true equilibrium points.

## Exercise

### 1) Derive the Equations of Motion

For simplicity we will assume that the Earth orbits the Sun in a circular orbit with exactly 1 AU between their centers of mass. It turns out that this assumption is not actually necessary, but it does make things conceptually simpler.

To find the Lagrange points, let's define a coordinate system with the origin at the center of mass of the Earth-Sun system. Following the figure above, we will assume that the Sun and Earth lie along the $x$-axis at $x_S = -R_S$ and $x_E = R_E$, respectively. 

Using Newton's Law of Gravitation, show that the acceleration experienced by a small test mass located at point $(x, y)$ is given by

$$ a_x (x, y) = - \frac{G M_S (x + R_S)}{((x + R_S)^2 + y^2 )^{3/2}}  - \frac{G M_E (x - R_E)}{((x - R_E)^2 + y^2 )^{3/2}} $$ 
$$ a_y (x, y) = - \frac{G M_S y}{((x + R_S)^2 + y^2 )^{3/2}}  - \frac{G M_E y}{((x - R_E)^2 + y^2 )^{3/2}} $$

To rotate synchronously with the Earth, the test mass should have the same rotational frequency $\omega$ as the Earth.  The centripetal force required to produce such motion is given by
$$\vec{a} = -\omega^2 \vec{r} = -\omega^2\left[ x\hat{x}+y\hat{y} \right] $$ 

If we equate this expression to the acceleration components derived above, we can solve for the position of the Lagrange points.  Show that the system of non-linear equations is

$$ - \frac{G M_S (x + R_S)}{((x + R_S)^2 + y^2 )^{3/2}}  - \frac{G M_E (x - R_E)}{((x - R_E)^2 + y^2 )^{3/2}} + \omega^2 x =0\tag{3}$$ 
and
$$ - \frac{G M_S y}{((x + R_S)^2 + y^2 )^{3/2}}  - \frac{G M_E y}{((x - R_E)^2 + y^2 )^{3/2}} + \omega^2 y = 0.\tag{4}$$

### 2) Plot the Equations of Motion

$\def\qty#1#2{#1\,\text{#2}}$

Equations 3 and 4 are now in a form suitable for application of Newton's method.  However, we also need a reasonable initial guess for the position of the Lagrange points.  To choose an initial guess, it is often useful to graph the functions we wish to solve and find where they are approximately zero.  And to do this, it is useful to have functions that calculate the left-hand side of Equations 3 and 4 **using units of years for time, astronomical units (AU) for distance, and solar masses ($M_\odot$) for mass**.  In these units, the universal gravitational constant is $\qty{4\pi^2}{AU$^3$ yr$^{-2}$ M$_\odot$}$, the mass of the Sun (Earth) is $\qty{1}{M$_\odot$}$ ($\qty{3.00\times10^{-6}}{M$_\odot$}$), and the angular velocity of the Earth is $\qty{2\pi}{rad/yr}$.  

To define such functions, you will need the position of the Earth and Sun.  The distance between the center of the Sun and the center of the Earth is defined as 1 AU. If the Sun and Earth lie on the $x$-axis with the center-of-mass at the origin, show that
$$R_S = \frac{M_E}{M_S + M_E}$$
and
$$R_E = \frac{M_S}{M_S + M_E}.$$

We now have everything we need to plot the left-hand side of Equations 3 and 4.  Define functions `fx(x, y)` and `fy(x, y)` to calculate these quantities and plot the results.  Also plot the magnitude of the vector $[f_x(x,y),f_y(x,y)]$.  Focus on the region with $x=\qty{1\pm.02}{AU}$ and $y=\qty{0\pm.02}{AU}$.  Can you identify Lagrange points 1 and 2 in your plot?

### 3) Find Lagrange Points with $y=0$

Let's first consider the Lagrange points with $y=0$.  In this case, Equation 4 becomes trivial and Equation 3 becomes
$$ - \frac{G M_S (x + R_S)}{((x + R_S)^2)^{3/2}}  - \frac{G M_E (x - R_E)}{((x - R_E)^2)^{3/2}} + \omega^2 x =0.$$ 

We can't simplify the denominators in this expression as one might expect, because, for example, $\left(\left(x+R_S\right)^2\right)^{3/2} \ne \left(x+R_S\right)^3$.  The first expression is positive definite, while the latter is not.  However, $\left(\left(x+R_S\right)^2\right)^{3/2} = \left|x+R_S\right|^3$, so we can simplify this to 
$$ - \frac{G M_S (x + R_S)}{\left|x + R_S\right|^3}  - \frac{G M_E (x - R_E)}{\left|x - R_E\right|^3} + \omega^2 x =0\tag{5}.$$ 

Equation 5 represents a single non-linear equation of the form $f(x)=0$, which we can solve using Newton's method to find the Lagrange points with $y=0$.  To do so, we also need the derivative of $f(x)$ with respect to $x$, which is given by

$$\frac{\mathrm{d}f}{\mathrm{d}x} = \frac{2G M_S}{\left|x + R_S\right|^3}  - \frac{2G M_E}{\left|x - R_E\right|^3} + \omega^2\tag{6}.$$

Write functions that calculate $f(x)$ and $\frac{\mathrm{d}f}{\mathrm{d}x}$ for any value of $x$, then use Newton's method to find the location of L1 and L2.  Finally, plot $f(x)$ to confirm that your algorithm found the correct roots.

Bonus: can you find L3?

### 4) Find Lagrange Points with $y\ne0$

The last two Lagrange points do not lie along the $x$-axis, so we need to simultaneously find roots of both $f_x(x,y)$ and $f_x(x,y)$. This requires the vector form of Newton's method to solve, as described above.  

The vector form of Newton's method requires knowledge of the Jacobian, defined in Equation 1.  Since these derivatives are cumbersome to calculate and implement, you may use this function to calculate the Jacobian:

In [1]:
def Jacobian(z):
    x=z[0]
    y=z[1]
    
    D_sun   = ((x+R_sun)**2   + y**2)**(1/2)
    D_earth = ((x-R_earth)**2 + y**2)**(1/2)
    
    J = np.empty((2,2))    
    J[0,0] = (-G*M_earth*(-R_earth + x)*(3.0*R_earth - 3.0*x)/D_earth**5 
                    - G*M_earth/D_earth**3 
                    - G*M_sun*(-3.0*R_sun - 3.0*x)*(R_sun + x)/D_sun**5 
                    - G*M_sun/D_sun**3 + omega**2)
    J[0,1] = (3.0*G*M_earth*y*(-R_earth + x)/D_earth**5 
                    + 3.0*G*M_sun*y*(R_sun + x)/D_sun**5)
    J[1,0] = (-G*M_earth*y*(3.0*R_earth - 3.0*x)/D_earth**5
                    - G*M_sun*y*(-3.0*R_sun - 3.0*x)/D_sun**5)
    J[1,1] = (3.0*G*M_earth*y**2/D_earth**5 
                    - G*M_earth/D_earth**3
                    + 3.0*G*M_sun*y**2/D_sun**5 
                    - G*M_sun/D_sun**3 + omega**2)

    return J

This function takes a $\vec{z}=[x,y]$ value as argument, and returns the Jacobian matrix.

Define a function `F(z)` which takes a $\vec{z}$ value as argument and returns the vector $F(z)$.

Finally, use the vector version of Newton's method to find the location of L4 and L5.

L4 and L5 can be shown to occur at $\left[ \cos (\pi/3), \pm \sin (\pi/3) \right]$.  Do your values agree?

### 5) Find Lagrange Points with `scipy`

The function `scipy.optimize.root` can be used to find roots.  Documentation on this function is available [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root.html).

Review the documentation on this function and use it to find all 5 Lagrange points.  You may have to experiment with different solving methods, by using the `method` argument.

### 6) Lagrange Point Stability

Having found the five Lagrange points, which are effective equilibria, the logical next step is to determine if these locations are stable or unstable equilibria. This can be done using the eigenvalues of the Jacobian matrix evaluated at each Lagrange point. In this case, all eigenvalues for stable equilibria will be positive, meaning that the "slope" of the potential is upward in all directions, while any negative eigenvalues indicate that the effective potential slopes downward along some direction. That means there exists at least one axis along which the object will accelerate away from the Lagrange point, and it is an unstable equilibrium point. 

Find the eigenvalues  of the Jacobian evaluated at each Lagrange point.  You may use an external library to find the eigenvalues.  Do the Lagrange points represent stable equilibria?

In units of yr$^{-2}$, the eigenvalues should be:
* L1: 360.13 and -120.85
* L2: 350.61 and -116.09
* L3: 118.44 and $-1.0373 \times 10^{-4}$
* L4: $2.6674 \times 10^{-4}$ and 118.43
* L5: $2.6674 \times 10^{-4}$ and 118.43

Though it is beyond the scope of this exercise, it can be shown that while L1, L2, and L3 are unstable, a spacecraft can "orbit" those points and be dynamically stable. For more on dynamic stability, see Butikov, Eugene I., 2001, "On the dynamic stabilization of an inverted pendulum", American Journal of Physics 69:7, 755-768.